# 4.20 — UMAP

UMAP is a way to turn unlabeled high-dimensional data into a small map by deciding which points are near each other, turning those decisions into a weighted graph, and reading geometry from that graph. In this lesson we build the graph-and-Laplacian core from scratch with NumPy, because the same math explains why scaling, neighbor choices, disconnected components, and stability checks matter.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build UMAP's graph view one idea at a time. Run each cell in order and inspect every small matrix. The walkthrough is self-contained and uses a `_w` suffix so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

### 1. Data become geometry only after we choose a distance

UMAP starts from measured coordinates, but the algorithm never sees a label or a human story. It sees distances. That makes scaling a modeling decision: a feature with larger units can dominate Euclidean distance and therefore dominate the graph.

In [ ]:
X_w = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
labels_w = ["A", "B", "C", "D"]
print("X shape:", X_w.shape)
print(X_w)

▶ What you'll see: four points in two measured coordinates, arranged as two vertical pairs.

In [ ]:
diff_w = X_w[:, None, :] - X_w[None, :, :]
D_w = np.sqrt(np.sum(diff_w ** 2, axis=2))
print("pairwise distances:\n", np.round(D_w, 3))
assert D_w.shape == (4, 4)

▶ What you'll see: nearby vertical neighbors are distance 1, while cross-pair neighbors are distance 4 or more.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_w[:, 0], X_w[:, 1], s=90, color="teal")
for i_w, name_w in enumerate(labels_w):
    plt.text(X_w[i_w, 0] + 0.06, X_w[i_w, 1] + 0.03, name_w)
plt.title("1: data as points before graph building")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: the eye sees two close pairs, but UMAP will only know that through distances.

*Why it's done this way:* a distance matrix is the audit trail for the rest of the method. If one feature has a huge numeric scale, its squared differences dominate the sum inside Euclidean distance, so the neighbor graph becomes a scale artifact rather than a manifold clue.

### 2. Distances become a weighted affinity graph

The next idea is to replace all pairwise distances with a local graph. For each point we keep its nearest neighbor and store an affinity weight. A simple Gaussian weight, $\exp(-d^2/\sigma^2)$, makes closer points heavier and farther points lighter.

In [ ]:
sigma_w = 1.5
A_w = np.zeros((4, 4))
for i_w in range(4):
    order_w = np.argsort(D_w[i_w])
    nbr_w = order_w[1]
    A_w[i_w, nbr_w] = np.exp(-(D_w[i_w, nbr_w] ** 2) / sigma_w ** 2)
A_w = np.maximum(A_w, A_w.T)
print("affinity graph A:\n", np.round(A_w, 3))

▶ What you'll see: A connects A-B and C-D with equal positive weights, and leaves the cross-pair edges at zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_w, cmap="viridis")
plt.colorbar(label="affinity")
plt.xticks(range(4), labels_w)
plt.yticks(range(4), labels_w)
plt.title("2: weighted nearest-neighbor graph A")
plt.show()

▶ What you'll see: two bright off-diagonal blocks and dark cross-block entries.

*Why it's done this way:* the graph is the algorithm's definition of local structure. Keeping local edges protects small neighborhoods from being washed out by global distances, while the exponential weight keeps the graph differentiable in spirit: a distance change becomes a smooth weight change.

### 3. Degrees summarize how much graph mass touches each point

Once we have affinities, the degree of a point is the sum of the weights connected to it. The diagonal degree matrix $D$ is bookkeeping, but it is essential because a highly connected point should not be treated the same as an isolated one.

In [ ]:
deg_w = A_w.sum(axis=1)
Deg_w = np.diag(deg_w)
print("degrees:", np.round(deg_w, 3))
print("D matrix:\n", np.round(Deg_w, 3))
assert np.allclose(deg_w, deg_w[0])

▶ What you'll see: all four degrees match because this toy graph contains two identical two-node components.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(labels_w, deg_w, color="orange")
plt.title("3: graph degree per point")
plt.ylabel("sum of incident affinities")
plt.show()

▶ What you'll see: every point has the same graph mass in this symmetric example.

*Why it's done this way:* degrees normalize the raw adjacency information into conservation accounting. The Laplacian subtracts adjacency from degree, so each row measures how different a point is from the weighted average of its graph neighbors.

### 4. The Laplacian turns a graph into a smoothness operator

The unnormalized graph Laplacian is $L=D-A$. It is small on vectors that assign similar values to connected points and large on vectors that jump across strong edges. This is the bridge from graph structure to coordinates.

In [ ]:
L_w = Deg_w - A_w
print("Laplacian L = D - A:\n", np.round(L_w, 3))
row_sums_w = L_w.sum(axis=1)
print("row sums:", np.round(row_sums_w, 6))
assert np.allclose(row_sums_w, 0.0)

▶ What you'll see: each row sums to zero, which is why a constant vector is always a zero-eigenvalue direction.

In [ ]:
z_w = np.array([1.0, 1.0, -1.0, -1.0])
energy_w = float(z_w @ L_w @ z_w)
print("smoothness energy z^T L z:", round(energy_w, 3))
assert round(energy_w, 3) == 0.0

▶ What you'll see: assigning one constant value per disconnected component costs zero energy.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(L_w, cmap="coolwarm")
plt.colorbar(label="L value")
plt.xticks(range(4), labels_w)
plt.yticks(range(4), labels_w)
plt.title("4: Laplacian matrix")
plt.show()

▶ What you'll see: positive diagonal mass is balanced by negative neighbor entries.

*Why it's done this way:* $z^T L z$ equals a weighted sum of squared differences across edges. Minimizing that energy gives coordinates that vary slowly along strong graph edges, exactly the notion of preserving local neighborhoods.

### 5. Eigenvectors become low-dimensional coordinates

Solving $Lv=\lambda v$ finds directions ordered by graph smoothness. The smallest eigenvalue is zero for the constant vector. The next nontrivial eigenvectors provide coordinates that separate graph components or slowly varying manifold directions.

In [ ]:
evals_w, evecs_w = np.linalg.eigh(L_w)
print("eigenvalues:", np.round(evals_w, 3))
zero_count_w = int(np.sum(np.isclose(evals_w, 0.0, atol=1e-8)))
print("zero eigenvalues:", zero_count_w)
assert np.allclose(np.round(evals_w, 3), [0.0, 0.0, round(2 * deg_w[0], 3), round(2 * deg_w[0], 3)])

▶ What you'll see: two zero eigenvalues, matching the two disconnected graph components.

In [ ]:
Z_w = evecs_w[:, :2]
print("first two eigenvector coordinates:\n", np.round(Z_w, 3))
print("Z shape:", Z_w.shape)
assert Z_w.shape == (4, 2)

▶ What you'll see: a two-column representation for four points, with one zero direction per component.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_w[:, 0], Z_w[:, 1], s=90, color="seagreen")
for i_w, name_w in enumerate(labels_w):
    plt.text(Z_w[i_w, 0] + 0.02, Z_w[i_w, 1] + 0.02, name_w)
plt.title("5: eigenvectors as graph coordinates")
plt.xlabel("eigenvector 0")
plt.ylabel("eigenvector 1")
plt.show()

▶ What you'll see: points in the same connected component share the same zero-energy coordinate pattern.

*Why it's done this way:* eigenvectors solve the constrained smoothness problem exactly for this linear graph objective. Small eigenvalues mean a coordinate can change without crossing strong edges, so they reveal components and broad geometry before noisy high-frequency directions.

### 6. Hyperparameters are lenses, so stability must be checked

Changing the neighbor count changes the graph, and changing the graph changes the embedding. UMAP outputs should therefore be read as a lens on structure, not as ground truth. A quick stability check compares nearby choices.

In [ ]:
D2_w = D_w.copy()
np.fill_diagonal(D2_w, np.inf)
A1_w = np.zeros((4, 4))
A2_w = np.zeros((4, 4))
for i_w in range(4):
    one_w = np.argsort(D2_w[i_w])[:1]
    two_w = np.argsort(D2_w[i_w])[:2]
    A1_w[i_w, one_w] = 1.0
    A2_w[i_w, two_w] = 1.0
A1_w = np.maximum(A1_w, A1_w.T)
A2_w = np.maximum(A2_w, A2_w.T)
print("edges with k=1:", int(A1_w.sum() / 2), "edges with k=2:", int(A2_w.sum() / 2))

▶ What you'll see: the larger neighborhood adds cross-pair edges, so the graph lens changes.

In [ ]:
L1_w = np.diag(A1_w.sum(axis=1)) - A1_w
L2_w = np.diag(A2_w.sum(axis=1)) - A2_w
e1_w = np.linalg.eigvalsh(L1_w)
e2_w = np.linalg.eigvalsh(L2_w)
print("k=1 eigenvalues:", np.round(e1_w, 3))
print("k=2 eigenvalues:", np.round(e2_w, 3))
assert int(np.sum(np.isclose(e1_w, 0.0))) == 2
assert int(np.sum(np.isclose(e2_w, 0.0))) == 1

▶ What you'll see: k=1 has two components, while k=2 connects the whole graph into one component.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(np.sort(e1_w), marker="o", label="k=1")
plt.plot(np.sort(e2_w), marker="s", label="k=2")
plt.title("6: spectrum changes with neighborhood size")
plt.xlabel("eigenvalue index")
plt.ylabel("eigenvalue")
plt.legend()
plt.show()

▶ What you'll see: the number of near-zero eigenvalues changes when the graph becomes connected.

*Why it's done this way:* stability checks ask whether the discovered structure survives nearby modeling choices. If a tiny change in k flips components or coordinates, the map may be a fragile artifact of the criterion rather than a robust pattern in the data.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Distances are the graph's raw material

UMAP's graph starts with distances. Four tiny points already show two close local pairs and larger
cross-pair gaps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_X = np.array([[0.0, 0.0],
                 [0.0, 1.0],
                 [3.0, 0.0],
                 [3.0, 1.0]])
t1_diff = t1_X[:, None, :] - t1_X[None, :, :]
t1_sq = t1_diff ** 2
t1_D = np.sqrt(t1_sq.sum(axis=2))
print("points:", t1_X.tolist())  # -> [[0.0, 0.0], [0.0, 1.0], [3.0, 0.0], [3.0, 1.0]]
print("distance matrix:", np.round(t1_D, 3).tolist())  # -> [[0.0, 1.0, 3.0, 3.162], [1.0, 0.0, 3.162, 3.0], [3.0, 3.162, 0.0, 1.0], [3.162, 3.0, 1.0, 0.0]]
assert round(float(t1_D[0, 1]), 3) == 1.0

plt.figure(figsize=(4.2, 3.2))
plt.imshow(t1_D, cmap="magma")
plt.colorbar(label="distance")
plt.title("Toy 1 · pairwise distances")
plt.xlabel("point j")
plt.ylabel("point i")
plt.show()

▶ What you'll see: the within-pair distances are 1, while cross-pair distances are about 3 or more.

### ✍️ Toy 2 · k-nearest neighbors turn distances into edges

Keeping `k=1` nearest neighbor per point converts the distance matrix into a local graph.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_D = np.array([[0.0, 1.0, 3.0, 3.162],
                 [1.0, 0.0, 3.162, 3.0],
                 [3.0, 3.162, 0.0, 1.0],
                 [3.162, 3.0, 1.0, 0.0]])
t2_masked = t2_D.copy()
np.fill_diagonal(t2_masked, np.inf)
t2_neighbors = np.argsort(t2_masked, axis=1)[:, 0]
t2_A_directed = np.zeros_like(t2_D)
t2_A_directed[np.arange(4), t2_neighbors] = 1.0
t2_A = np.maximum(t2_A_directed, t2_A_directed.T)
t2_edge_count = int(t2_A.sum() / 2)
print("nearest neighbors:", t2_neighbors.tolist())  # -> [1, 0, 3, 2]
print("directed graph:", t2_A_directed.astype(int).tolist())  # -> [[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]]
print("symmetric graph:", t2_A.astype(int).tolist())  # -> [[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]]
print("edge count:", t2_edge_count)  # -> 2
assert t2_edge_count == 2

plt.figure(figsize=(4.0, 3.0))
plt.imshow(t2_A, cmap="Greens")
plt.title("Toy 2 · k=1 adjacency")
plt.xlabel("point j")
plt.ylabel("point i")
plt.show()

▶ What you'll see: the graph has exactly two undirected edges, one for each close pair.

### ✍️ Toy 3 · A Gaussian affinity weights each kept edge

UMAP-like graph weights should be stronger for smaller distances. A simple Gaussian affinity makes
distance 1 become weight about 0.641 when sigma is 1.5.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_D = np.array([[0.0, 1.0, 3.0, 3.162],
                 [1.0, 0.0, 3.162, 3.0],
                 [3.0, 3.162, 0.0, 1.0],
                 [3.162, 3.0, 1.0, 0.0]])
t3_A_binary = np.array([[0.0, 1.0, 0.0, 0.0],
                        [1.0, 0.0, 0.0, 0.0],
                        [0.0, 0.0, 0.0, 1.0],
                        [0.0, 0.0, 1.0, 0.0]])
t3_sigma = 1.5
t3_weights = np.exp(-(t3_D ** 2) / (t3_sigma ** 2))
t3_A = t3_A_binary * t3_weights
print("sigma:", t3_sigma)  # -> 1.5
print("kept-edge weight:", round(float(t3_A[0, 1]), 3))  # -> 0.641
print("weighted affinity:", np.round(t3_A, 3).tolist())  # -> [[0.0, 0.641, 0.0, 0.0], [0.641, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.641], [0.0, 0.0, 0.641, 0.0]]
assert round(float(t3_A[0, 1]), 3) == 0.641

plt.figure(figsize=(4.0, 3.0))
plt.imshow(t3_A, cmap="viridis")
plt.colorbar(label="affinity")
plt.title("Toy 3 · weighted local graph")
plt.xlabel("point j")
plt.ylabel("point i")
plt.show()

▶ What you'll see: only the local edges are bright, and both close pairs receive the same weight.

### ✍️ Toy 4 · Degrees and the Laplacian balance graph mass

The degree of each point is the sum of incident affinity weights. The Laplacian `L = D - A` subtracts
neighbor mass from diagonal mass.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_w = np.exp(-1.0 / (1.5 ** 2))
t4_A = np.array([[0.0, t4_w, 0.0, 0.0],
                 [t4_w, 0.0, 0.0, 0.0],
                 [0.0, 0.0, 0.0, t4_w],
                 [0.0, 0.0, t4_w, 0.0]])
t4_deg = t4_A.sum(axis=1)
t4_Deg = np.diag(t4_deg)
t4_L = t4_Deg - t4_A
t4_row_sums = t4_L.sum(axis=1)
print("degrees:", np.round(t4_deg, 3).tolist())  # -> [0.641, 0.641, 0.641, 0.641]
print("Laplacian:", np.round(t4_L, 3).tolist())  # -> [[0.641, -0.641, 0.0, 0.0], [-0.641, 0.641, 0.0, 0.0], [0.0, 0.0, 0.641, -0.641], [0.0, 0.0, -0.641, 0.641]]
print("row sums:", np.round(t4_row_sums, 6).tolist())  # -> [0.0, 0.0, 0.0, 0.0]
assert np.allclose(t4_row_sums, 0.0)

plt.figure(figsize=(4.0, 3.0))
plt.imshow(t4_L, cmap="coolwarm")
plt.colorbar(label="L value")
plt.title("Toy 4 · L = D - A")
plt.xlabel("point j")
plt.ylabel("point i")
plt.show()

▶ What you'll see: positive diagonal degree is exactly balanced by negative neighbor entries.

### ✍️ Toy 5 · Smoothness energy penalizes jumps across edges

A coordinate that is constant within each connected pair pays zero energy. A coordinate that flips
inside each pair pays a positive penalty.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_w = np.exp(-1.0 / (1.5 ** 2))
t5_A = np.array([[0.0, t5_w, 0.0, 0.0],
                 [t5_w, 0.0, 0.0, 0.0],
                 [0.0, 0.0, 0.0, t5_w],
                 [0.0, 0.0, t5_w, 0.0]])
t5_L = np.diag(t5_A.sum(axis=1)) - t5_A
t5_z_smooth = np.array([1.0, 1.0, -1.0, -1.0])
t5_z_jump = np.array([1.0, -1.0, 1.0, -1.0])
t5_energy_smooth = float(t5_z_smooth @ t5_L @ t5_z_smooth)
t5_energy_jump = float(t5_z_jump @ t5_L @ t5_z_jump)
print("smooth coordinate:", t5_z_smooth.tolist())  # -> [1.0, 1.0, -1.0, -1.0]
print("jumping coordinate:", t5_z_jump.tolist())  # -> [1.0, -1.0, 1.0, -1.0]
print("smooth energy:", round(t5_energy_smooth, 3))  # -> 0.0
print("jump energy:", round(t5_energy_jump, 3))  # -> 5.129
assert round(t5_energy_smooth, 3) == 0.0
assert t5_energy_jump > t5_energy_smooth

plt.figure(figsize=(4.4, 2.8))
plt.bar(["constant on pairs", "jumps on pairs"], [t5_energy_smooth, t5_energy_jump], color=["seagreen", "crimson"])
plt.ylabel("zᵀLz")
plt.title("Toy 5 · graph smoothness energy")
plt.xticks(rotation=12)
plt.show()

▶ What you'll see: only the coordinate that changes across graph edges pays energy.

### ✍️ Toy 6 · Laplacian eigenvalues count components

Zero eigenvalues reveal connected components. The next eigenvectors become graph coordinates.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_w = np.exp(-1.0 / (1.5 ** 2))
t6_A = np.array([[0.0, t6_w, 0.0, 0.0],
                 [t6_w, 0.0, 0.0, 0.0],
                 [0.0, 0.0, 0.0, t6_w],
                 [0.0, 0.0, t6_w, 0.0]])
t6_L = np.diag(t6_A.sum(axis=1)) - t6_A
t6_evals, t6_evecs = np.linalg.eigh(t6_L)
t6_zero_count = int(np.sum(np.isclose(t6_evals, 0.0)))
t6_coords = t6_evecs[:, :2]
print("eigenvalues:", np.round(t6_evals, 3).tolist())  # -> [0.0, 0.0, 1.282, 1.282]
print("zero eigenvalues:", t6_zero_count)  # -> 2
print("first two coordinates:", np.round(t6_coords, 3).tolist())  # -> [[-0.707, -0.0], [-0.707, -0.0], [-0.0, -0.707], [-0.0, -0.707]]
assert t6_zero_count == 2

plt.figure(figsize=(4.0, 3.0))
plt.scatter(t6_coords[:, 0], t6_coords[:, 1], s=90, color="purple")
plt.title("Toy 6 · eigenvectors as graph coordinates")
plt.xlabel("eigenvector 0")
plt.ylabel("eigenvector 1")
plt.show()

▶ What you'll see: each disconnected pair shares a zero-energy coordinate pattern.

### ✍️ Toy 7 · Neighbor count changes connectivity

Changing `k` changes the graph lens. Here `k=1` leaves two components, while `k=2` connects the
rectangle.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_X = np.array([[0.0, 0.0],
                 [0.0, 1.0],
                 [3.0, 0.0],
                 [3.0, 1.0]])
t7_D = np.sqrt(np.sum((t7_X[:, None, :] - t7_X[None, :, :]) ** 2, axis=2))
t7_edges = []
t7_zero_counts = []
for t7_k in [1, 2]:
    t7_masked = t7_D.copy()
    np.fill_diagonal(t7_masked, np.inf)
    t7_A = np.zeros_like(t7_D)
    for t7_i in range(t7_D.shape[0]):
        t7_nbrs = np.argsort(t7_masked[t7_i])[:t7_k]
        t7_A[t7_i, t7_nbrs] = 1.0
    t7_A = np.maximum(t7_A, t7_A.T)
    t7_L = np.diag(t7_A.sum(axis=1)) - t7_A
    t7_evals = np.linalg.eigvalsh(t7_L)
    t7_edges.append(int(t7_A.sum() / 2))
    t7_zero_counts.append(int(np.sum(np.isclose(t7_evals, 0.0))))
    print("k", t7_k, "edges", int(t7_A.sum() / 2), "eigenvalues", np.round(t7_evals, 3).tolist())
# -> k 1 edges 2 eigenvalues [0.0, 0.0, 2.0, 2.0]
# -> k 2 edges 4 eigenvalues [-0.0, 2.0, 2.0, 4.0]
assert t7_zero_counts == [2, 1]

plt.figure(figsize=(4.4, 2.8))
plt.bar(["k=1", "k=2"], t7_zero_counts, color=["crimson", "seagreen"])
plt.ylabel("zero eigenvalues")
plt.title("Toy 7 · k changes components")
plt.show()

▶ What you'll see: increasing k removes one zero eigenvalue by connecting the graph.

### ✍️ Toy 8 · Sigma controls how strongly edges pull

The same distance receives different affinity weights depending on the kernel width sigma.

In [ ]:
import numpy as np

t8_rng = np.random.default_rng(0)
t8_sigmas = np.array([0.5, 1.5, 3.0])
t8_near_d = 1.0
t8_far_d = 3.0
t8_near_weights = np.exp(-(t8_near_d ** 2) / (t8_sigmas ** 2))
t8_far_weights = np.exp(-(t8_far_d ** 2) / (t8_sigmas ** 2))
print("sigmas:", t8_sigmas.tolist())  # -> [0.5, 1.5, 3.0]
print("weight at distance 1:", np.round(t8_near_weights, 3).tolist())  # -> [0.018, 0.641, 0.895]
print("weight at distance 3:", np.round(t8_far_weights, 3).tolist())  # -> [0.0, 0.018, 0.368]
assert t8_near_weights[-1] > t8_near_weights[0]
assert t8_far_weights[-1] > t8_far_weights[0]

plt.figure(figsize=(4.4, 2.8))
plt.plot(t8_sigmas, t8_near_weights, marker="o", label="d=1")
plt.plot(t8_sigmas, t8_far_weights, marker="s", label="d=3")
plt.xlabel("sigma")
plt.ylabel("affinity")
plt.legend()
plt.title("Toy 8 · wider sigma keeps more mass")
plt.show()

▶ What you'll see: wider sigma makes both near and far edges heavier, especially the far one.

### ✍️ Toy 9 · Scaling can change nearest neighbors

UMAP's graph depends on Euclidean distance. Standardizing features can change which edge is considered
local.

In [ ]:
import numpy as np

t9_rng = np.random.default_rng(0)
t9_X = np.array([[0.0, 0.0],
                 [0.1, 100.0],
                 [0.2, 200.0],
                 [2.0, 210.0]])
t9_D_raw = np.sqrt(np.sum((t9_X[:, None, :] - t9_X[None, :, :]) ** 2, axis=2))
t9_X_std = (t9_X - t9_X.mean(axis=0)) / t9_X.std(axis=0)
t9_D_std = np.sqrt(np.sum((t9_X_std[:, None, :] - t9_X_std[None, :, :]) ** 2, axis=2))
t9_raw_masked = t9_D_raw.copy()
t9_std_masked = t9_D_std.copy()
np.fill_diagonal(t9_raw_masked, np.inf)
np.fill_diagonal(t9_std_masked, np.inf)
t9_raw_nn = int(np.argmin(t9_raw_masked[2]))
t9_std_nn = int(np.argmin(t9_std_masked[2]))
print("raw distances from point 2:", np.round(t9_D_raw[2], 3).tolist())  # -> [200.0, 100.0, 0.0, 10.161]
print("standardized distances from point 2:", np.round(t9_D_std[2], 3).tolist())  # -> [2.358, 1.179, 0.0, 2.183]
print("raw nearest / standardized nearest:", t9_raw_nn, t9_std_nn)  # -> 3 1
assert t9_raw_nn == 3
assert t9_std_nn == 1

plt.figure(figsize=(4.8, 2.8))
plt.bar(["raw → p3", "standardized → p1"], [t9_D_raw[2, 3], t9_D_std[2, 1]], color=["crimson", "steelblue"])
plt.ylabel("chosen neighbor distance")
plt.title("Toy 9 · scale changes the local edge")
plt.xticks(rotation=12)
plt.show()

▶ What you'll see: the nearest neighbor of point 2 flips after each feature is put on comparable scale.

### ✍️ Toy 10 · Align embeddings before comparing stability

Two spectral maps can be rotated. Procrustes alignment removes that arbitrary rotation before measuring
whether coordinates changed.

In [ ]:
import numpy as np

t10_rng = np.random.default_rng(0)
t10_Z = np.array([[-1.0, 0.0],
                  [-1.0, 0.2],
                  [1.0, 0.0],
                  [1.0, 0.2]])
t10_theta = np.pi / 2
t10_R = np.array([[np.cos(t10_theta), -np.sin(t10_theta)],
                  [np.sin(t10_theta), np.cos(t10_theta)]])
t10_Z_rot = t10_Z @ t10_R
t10_Zc = t10_Z - t10_Z.mean(axis=0)
t10_Zrot_c = t10_Z_rot - t10_Z_rot.mean(axis=0)
t10_U, t10_s, t10_Vt = np.linalg.svd(t10_Zrot_c.T @ t10_Zc)
t10_Q = t10_U @ t10_Vt
t10_aligned = t10_Zrot_c @ t10_Q
t10_error = float(np.linalg.norm(t10_Zc - t10_aligned))
print("singular values:", np.round(t10_s, 3).tolist())  # -> [4.0, 0.04]
print("alignment error:", round(t10_error, 6))  # -> 0.0
print("aligned coordinates:", np.round(t10_aligned, 3).tolist())  # -> [[-1.0, -0.1], [-1.0, 0.1], [1.0, -0.1], [1.0, 0.1]]
assert t10_error < 1e-10

plt.figure(figsize=(4.4, 3.0))
plt.scatter(t10_Zc[:, 0], t10_Zc[:, 1], s=80, label="reference")
plt.scatter(t10_aligned[:, 0], t10_aligned[:, 1], s=80, marker="x", label="aligned")
plt.legend()
plt.title("Toy 10 · alignment removes rotation")
plt.xlabel("axis 0")
plt.ylabel("axis 1")
plt.show()

▶ What you'll see: the aligned rotated map lands exactly on the centered reference map.

### ✍️ Toy 11 · Bootstrap graph stability compares edge sets

A stability check perturbs the data and asks whether the local graph edges survive.

In [ ]:
import numpy as np

t11_rng = np.random.default_rng(0)
t11_X = np.array([[0.0, 0.0],
                  [0.0, 1.0],
                  [3.0, 0.0],
                  [3.0, 1.0],
                  [6.0, 0.0],
                  [6.0, 1.0]])
t11_noise = 0.2 * t11_rng.normal(size=t11_X.shape)
t11_X_noisy = t11_X + t11_noise

def t11_edges(t11_points):
    t11_D = np.sqrt(np.sum((t11_points[:, None, :] - t11_points[None, :, :]) ** 2, axis=2))
    np.fill_diagonal(t11_D, np.inf)
    t11_A = np.zeros((t11_points.shape[0], t11_points.shape[0]), dtype=int)
    for t11_i in range(t11_points.shape[0]):
        t11_j = int(np.argsort(t11_D[t11_i])[0])
        t11_A[t11_i, t11_j] = 1
    t11_A = np.maximum(t11_A, t11_A.T)
    return sorted(tuple(t11_pair) for t11_pair in zip(*np.where(np.triu(t11_A, 1) > 0)))

t11_edges_base = t11_edges(t11_X)
t11_edges_noisy = t11_edges(t11_X_noisy)
t11_intersection = len(set(t11_edges_base) & set(t11_edges_noisy))
t11_union = len(set(t11_edges_base) | set(t11_edges_noisy))
t11_jaccard = t11_intersection / t11_union
print("noisy points:", np.round(t11_X_noisy, 3).tolist())  # -> [[0.025, -0.026], [0.128, 1.021], [2.893, 0.072], [3.261, 1.189], [5.859, -0.253], [5.875, 1.008]]
print("base edges:", t11_edges_base)  # -> [(0, 1), (2, 3), (4, 5)]
print("noisy edges:", t11_edges_noisy)  # -> [(0, 1), (2, 3), (4, 5)]
print("edge Jaccard:", round(float(t11_jaccard), 3))  # -> 1.0
assert t11_jaccard == 1.0

plt.figure(figsize=(5.0, 2.8))
plt.scatter(t11_X[:, 0], t11_X[:, 1], s=80, label="base")
plt.scatter(t11_X_noisy[:, 0], t11_X_noisy[:, 1], s=80, marker="x", label="noisy")
plt.legend()
plt.title("Toy 11 · same nearest-neighbor edges")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: small noise leaves the three local pair edges unchanged in this stable toy.

### ✍️ Toy 12 · A weak bridge is a fallback for disconnected graphs

If the graph is disconnected, the Laplacian has multiple zero eigenvalues. Adding a weak bridge makes
one connected graph with one zero eigenvalue.

In [ ]:
import numpy as np

t12_rng = np.random.default_rng(0)
t12_A = np.array([[0.0, 1.0, 0.0, 0.0],
                  [1.0, 0.0, 0.0, 0.0],
                  [0.0, 0.0, 0.0, 1.0],
                  [0.0, 0.0, 1.0, 0.0]])
t12_L = np.diag(t12_A.sum(axis=1)) - t12_A
t12_evals = np.linalg.eigvalsh(t12_L)
t12_A_bridge = t12_A.copy()
t12_A_bridge[1, 2] = 0.1
t12_A_bridge[2, 1] = 0.1
t12_L_bridge = np.diag(t12_A_bridge.sum(axis=1)) - t12_A_bridge
t12_evals_bridge = np.linalg.eigvalsh(t12_L_bridge)
t12_zeros_before = int(np.sum(np.isclose(t12_evals, 0.0)))
t12_zeros_after = int(np.sum(np.isclose(t12_evals_bridge, 0.0)))
print("eigenvalues before bridge:", np.round(t12_evals, 3).tolist())  # -> [0.0, 0.0, 2.0, 2.0]
print("eigenvalues after bridge:", np.round(t12_evals_bridge, 3).tolist())  # -> [0.0, 0.095, 2.0, 2.105]
print("zero counts before/after:", t12_zeros_before, t12_zeros_after)  # -> 2 1
assert t12_zeros_before == 2
assert t12_zeros_after == 1

plt.figure(figsize=(4.4, 2.8))
plt.bar(["before", "after weak bridge"], [t12_zeros_before, t12_zeros_after], color=["crimson", "seagreen"])
plt.ylabel("zero eigenvalues")
plt.title("Toy 12 · bridge fixes disconnected graph")
plt.show()

▶ What you'll see: the extra weak edge removes the duplicate zero eigenvalue while keeping a small spectral gap.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

def pairwise_distances(X):
    diff = X[:, None, :] - X[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

def knn_affinity(X, k=1, sigma=1.0):
    D = pairwise_distances(X)
    A = np.zeros((X.shape[0], X.shape[0]))
    masked = D.copy()
    np.fill_diagonal(masked, np.inf)
    for i in range(X.shape[0]):
        nbrs = np.argsort(masked[i])[:k]
        A[i, nbrs] = np.exp(-(D[i, nbrs] ** 2) / (sigma ** 2))
    return np.maximum(A, A.T)

def laplacian(A):
    return np.diag(A.sum(axis=1)) - A

def spectral_coordinates(A, dim=2):
    vals, vecs = np.linalg.eigh(laplacian(A))
    return vals, vecs[:, :dim]

## 🟢 Basics (warm-up)

### Basic 1 — Make a tiny unlabeled dataset

**Goal.** Create points with no labels used by the algorithm, because UMAP-style learning starts from geometry rather than targets. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[0.0, 0.0], [0.2, 0.1], [3.0, 0.0], [3.2, 0.1]])
print("shape:", X_b1.shape)
print(X_b1)
assert X_b1.shape == (4, 2)

▶ What you'll see: a 4×2 matrix, meaning four examples and two measured coordinates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_b1[:, 0], X_b1[:, 1], s=90, color="teal")
plt.title("Basic 1: unlabeled points")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: two visible pairs, but no class labels are provided to the code.

👀 Takeaway: UMAP begins with coordinates and a distance choice, not a teacher signal.

### Basic 2 — Compute pairwise distances

**Goal.** Build the full distance matrix, because neighborhood graphs are chosen from pairwise closeness. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
D_b2 = pairwise_distances(X_b2)
print("distances:\n", np.round(D_b2, 3))
assert round(float(D_b2[0, 1]), 3) == 1.0

▶ What you'll see: the matrix is symmetric and has zeros on the diagonal.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(D_b2, cmap="magma")
plt.colorbar(label="distance")
plt.title("Basic 2: pairwise distances")
plt.show()

▶ What you'll see: small distances are concentrated within the two close pairs.

👀 Takeaway: every later graph decision is downstream of this numeric distance table.

### Basic 3 — Find one nearest neighbor per point

**Goal.** Select local neighbors, because UMAP preserves local neighborhoods rather than every long-range distance. We build it in 2 steps.

In [ ]:
D_b3 = pairwise_distances(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]))
masked_b3 = D_b3.copy()
np.fill_diagonal(masked_b3, np.inf)
nn_b3 = np.argmin(masked_b3, axis=1)
print("nearest neighbor index per point:", nn_b3)
assert np.array_equal(nn_b3, np.array([1, 0, 3, 2]))

▶ What you'll see: each point chooses the other point in its close vertical pair.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(4), masked_b3[np.arange(4), nn_b3], color="orange")
plt.title("Basic 3: nearest-neighbor distances")
plt.xlabel("point index")
plt.ylabel("distance to nearest neighbor")
plt.show()

▶ What you'll see: all nearest-neighbor distances equal 1 in this symmetric toy case.

👀 Takeaway: the neighbor rule converts raw distances into local evidence.

### Basic 4 — Convert a distance to an affinity weight

**Goal.** Use an exponential kernel, because a smooth weight lets close points count more than far points. We build it in 2 steps.

In [ ]:
d_b4 = np.array([0.0, 1.0, 2.0, 4.0])
sigma_b4 = 2.0
w_b4 = np.exp(-(d_b4 ** 2) / sigma_b4 ** 2)
print("weights:", np.round(w_b4, 3))
assert round(float(w_b4[1]), 3) == 0.779

▶ What you'll see: weight is 1 at distance 0 and shrinks as distance grows.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(d_b4, w_b4, marker="o", color="seagreen")
plt.title("Basic 4: distance-to-weight curve")
plt.xlabel("distance")
plt.ylabel("affinity")
plt.show()

▶ What you'll see: far distances fade toward zero influence.

👀 Takeaway: affinities are soft neighbor strengths, not just yes-or-no edges.

### Basic 5 — Build a symmetric affinity matrix

**Goal.** Assemble the graph matrix A, because graph methods operate on point-to-point weights. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A_b5 = knn_affinity(X_b5, k=1, sigma=2.0)
print("A:\n", np.round(A_b5, 3))
assert np.allclose(A_b5, A_b5.T)

▶ What you'll see: the graph has two undirected edges with equal weights.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_b5, cmap="viridis")
plt.colorbar(label="affinity")
plt.title("Basic 5: symmetric affinity graph")
plt.show()

▶ What you'll see: the matrix has mirrored bright entries because the graph is undirected.

👀 Takeaway: symmetrizing makes either point's neighbor choice count as a shared graph edge.

### Basic 6 — Compute degrees

**Goal.** Sum graph weights at each point, because degrees tell the Laplacian how much mass each node owns. We build it in 2 steps.

In [ ]:
A_b6 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
deg_b6 = A_b6.sum(axis=1)
print("degrees:", np.round(deg_b6, 3))
assert np.allclose(deg_b6, deg_b6[0])

▶ What you'll see: each point has one edge of the same strength.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["0", "1", "2", "3"], deg_b6, color="purple")
plt.title("Basic 6: degree per point")
plt.ylabel("weighted degree")
plt.show()

▶ What you'll see: equal bars because the toy graph is balanced.

👀 Takeaway: degree is the local normalizer behind the Laplacian.

### Basic 7 — Form the Laplacian

**Goal.** Compute L = D - A, because this matrix measures graph smoothness. We build it in 2 steps.

In [ ]:
A_b7 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
L_b7 = laplacian(A_b7)
print("L:\n", np.round(L_b7, 3))
assert np.allclose(L_b7.sum(axis=1), 0.0)

▶ What you'll see: positive diagonal entries and negative neighbor entries balance to row sum zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(L_b7, cmap="coolwarm")
plt.colorbar(label="L value")
plt.title("Basic 7: graph Laplacian")
plt.show()

▶ What you'll see: each component appears as a small Laplacian block.

👀 Takeaway: the Laplacian turns graph edges into a matrix that penalizes rough coordinates.

### Basic 8 — Count connected components with eigenvalues

**Goal.** Inspect Laplacian eigenvalues, because zero eigenvalues count disconnected components. We build it in 2 steps.

In [ ]:
A_b8 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
vals_b8 = np.linalg.eigvalsh(laplacian(A_b8))
zero_count_b8 = int(np.sum(np.isclose(vals_b8, 0.0, atol=1e-8)))
print("eigenvalues:", np.round(vals_b8, 3), "zero count:", zero_count_b8)
assert zero_count_b8 == 2

▶ What you'll see: two zero eigenvalues reveal two disconnected graph components.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(vals_b8, marker="o", color="navy")
plt.title("Basic 8: Laplacian spectrum")
plt.xlabel("index")
plt.ylabel("eigenvalue")
plt.show()

▶ What you'll see: the spectrum begins with two zeros, then jumps upward.

👀 Takeaway: eigenvalues make graph connectivity measurable.

### Basic 9 — Use eigenvectors as coordinates

**Goal.** Extract low-dimensional graph coordinates, because smooth eigenvectors preserve strong local edges. We build it in 2 steps.

In [ ]:
A_b9 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
vals_b9, Z_b9 = spectral_coordinates(A_b9, dim=2)
print("Z:\n", np.round(Z_b9, 3))
assert Z_b9.shape == (4, 2)

▶ What you'll see: four rows of two graph-derived coordinates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_b9[:, 0], Z_b9[:, 1], s=90, color="crimson")
for i_b9 in range(4):
    plt.text(Z_b9[i_b9, 0] + 0.02, Z_b9[i_b9, 1] + 0.02, str(i_b9))
plt.title("Basic 9: eigenvector coordinates")
plt.xlabel("coord 0")
plt.ylabel("coord 1")
plt.show()

▶ What you'll see: points in the same component share coordinate structure.

👀 Takeaway: graph eigenvectors are a simple from-scratch embedding mechanism.

### Basic 10 — Check shape bookkeeping

**Goal.** Track matrix shapes, because most implementation bugs in embedding code are shape mismatches. We build it in 2 steps.

In [ ]:
X_b10 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A_b10 = knn_affinity(X_b10, k=1, sigma=2.0)
L_b10 = laplacian(A_b10)
vals_b10, Z_b10 = spectral_coordinates(A_b10, dim=1)
print("X", X_b10.shape, "A", A_b10.shape, "L", L_b10.shape, "Z", Z_b10.shape)
assert X_b10.shape == (4, 2) and Z_b10.shape == (4, 1)

▶ What you'll see: data are 4×2, graph matrices are 4×4, and the embedding is 4×1.

In [ ]:
plt.figure(figsize=(4, 2.5))
plt.scatter(Z_b10[:, 0], np.zeros(4), s=90, color="teal")
plt.yticks([])
plt.title("Basic 10: one-dimensional representation")
plt.xlabel("Z coordinate")
plt.show()

▶ What you'll see: the embedding keeps one coordinate per example.

👀 Takeaway: UMAP-like workflows transform example features into example embeddings, while graph matrices stay example by example.

## 🟡 Easy

### Easy 1 — Scaling can change neighbors

**Goal.** Show scale sensitivity, because distance-based graphs only see the numeric units we provide. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0.0, 0.0], [0.0, 2.0], [1.0, 0.1]])
D_raw_e1 = pairwise_distances(X_e1)
print("raw distances from point 0:", np.round(D_raw_e1[0], 3))

▶ What you'll see: point 2 is closer to point 0 than point 1 under the raw coordinates.

In [ ]:
X_scaled_e1 = X_e1.copy()
X_scaled_e1[:, 0] *= 5.0
D_scaled_e1 = pairwise_distances(X_scaled_e1)
print("scaled distances from point 0:", np.round(D_scaled_e1[0], 3))
assert int(np.argmin(np.where(np.arange(3) == 0, np.inf, D_raw_e1[0]))) == 2
assert int(np.argmin(np.where(np.arange(3) == 0, np.inf, D_scaled_e1[0]))) == 1

▶ What you'll see: after scaling feature 0, the nearest neighbor of point 0 flips.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_e1[:, 0], X_e1[:, 1], s=90, label="raw")
plt.scatter(X_scaled_e1[:, 0], X_scaled_e1[:, 1], s=90, marker="x", label="scaled")
plt.title("Easy 1: feature scale changes geometry")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.legend()
plt.show()

▶ What you'll see: stretching one axis changes which point is geometrically close.

👀 Takeaway: scaling is preprocessing, but it is also a modeling choice for UMAP.

### Easy 2 — Compare k-neighborhood graphs

**Goal.** Build graphs with k=1 and k=2, because neighborhood size controls local versus global connectivity. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A1_e2 = knn_affinity(X_e2, k=1, sigma=3.0)
A2_e2 = knn_affinity(X_e2, k=2, sigma=3.0)
print("edge counts:", int(np.sum(A1_e2 > 0) / 2), int(np.sum(A2_e2 > 0) / 2))

▶ What you'll see: k=2 has more graph edges than k=1.

In [ ]:
z1_e2 = int(np.sum(np.isclose(np.linalg.eigvalsh(laplacian(A1_e2)), 0.0, atol=1e-8)))
z2_e2 = int(np.sum(np.isclose(np.linalg.eigvalsh(laplacian(A2_e2)), 0.0, atol=1e-8)))
print("component counts:", z1_e2, z2_e2)
assert z1_e2 == 2 and z2_e2 == 1

▶ What you'll see: k=2 connects the two pairs into a single graph.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(A1_e2, cmap="viridis")
ax[0].set_title("k=1")
ax[1].imshow(A2_e2, cmap="viridis")
ax[1].set_title("k=2")
plt.suptitle("Easy 2: neighborhood size changes A")
plt.show()

▶ What you'll see: the k=2 matrix has extra bright cross-pair entries.

👀 Takeaway: k is a lens width, and the graph spectrum reveals its connectivity effect.

### Easy 3 — Make a one-dimensional spectral map

**Goal.** Embed a connected chain into one coordinate, because the second-smallest eigenvector often orders points along a graph. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
A_e3 = knn_affinity(X_e3, k=2, sigma=2.0)
vals_e3, vecs_e3 = np.linalg.eigh(laplacian(A_e3))
coord_e3 = vecs_e3[:, 1]
print("eigenvalues:", np.round(vals_e3, 3))
assert vals_e3[1] > 0

▶ What you'll see: one zero eigenvalue followed by a small positive smooth direction.

In [ ]:
print("1D coordinate:", np.round(coord_e3, 3))
order_e3 = np.argsort(coord_e3)
print("coordinate order:", order_e3)
assert set(order_e3.tolist()) == set(range(5))

▶ What you'll see: the eigenvector gives a monotone ordering up to sign.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(coord_e3, np.zeros_like(coord_e3), s=90, color="seagreen")
for i_e3 in range(5):
    plt.text(coord_e3[i_e3] + 0.01, 0.01, str(i_e3))
plt.yticks([])
plt.title("Easy 3: chain embedded in 1D")
plt.xlabel("spectral coordinate")
plt.show()

▶ What you'll see: neighboring chain points stay near each other in the coordinate.

👀 Takeaway: smooth graph eigenvectors can recover a simple manifold order.

### Easy 4 — Detect disconnected components

**Goal.** Count graph components from zero eigenvalues, because disconnected pieces cannot be smoothly arranged by a single connected coordinate. We build it in 3 steps.

In [ ]:
A_e4 = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                 [1.0, 0.0, 0.0, 0.0, 0.0],
                 [0.0, 0.0, 0.0, 1.0, 0.0],
                 [0.0, 0.0, 1.0, 0.0, 1.0],
                 [0.0, 0.0, 0.0, 1.0, 0.0]])
L_e4 = laplacian(A_e4)
vals_e4 = np.linalg.eigvalsh(L_e4)
print("eigenvalues:", np.round(vals_e4, 3))

▶ What you'll see: the graph has two separate blocks and therefore two zero eigenvalues.

In [ ]:
components_e4 = int(np.sum(np.isclose(vals_e4, 0.0, atol=1e-8)))
print("components:", components_e4)
assert components_e4 == 2

▶ What you'll see: the component count is read directly from the Laplacian spectrum.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_e4, cmap="Greens")
plt.title("Easy 4: two graph components")
plt.colorbar(label="edge")
plt.show()

▶ What you'll see: two disconnected adjacency blocks.

👀 Takeaway: zero eigenvalues are not numerical decoration; they diagnose disconnected structure.

### Easy 5 — Quantify graph smoothness

**Goal.** Compute $z^T L z$, because embeddings prefer coordinates that do not jump across strong edges. We build it in 3 steps.

In [ ]:
A_e5 = np.array([[0.0, 1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
L_e5 = laplacian(A_e5)
z_smooth_e5 = np.array([1.0, 1.0, 1.0])
z_rough_e5 = np.array([1.0, -1.0, 1.0])
print("L:\n", L_e5)

▶ What you'll see: the middle point connects to both endpoints.

In [ ]:
energy_smooth_e5 = float(z_smooth_e5 @ L_e5 @ z_smooth_e5)
energy_rough_e5 = float(z_rough_e5 @ L_e5 @ z_rough_e5)
print("energies:", energy_smooth_e5, energy_rough_e5)
assert energy_smooth_e5 == 0.0 and energy_rough_e5 == 8.0

▶ What you'll see: a constant coordinate costs zero, while a sign flip across edges costs a lot.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["smooth", "rough"], [energy_smooth_e5, energy_rough_e5], color=["teal", "red"])
plt.title("Easy 5: Laplacian smoothness energy")
plt.ylabel("z^T L z")
plt.show()

▶ What you'll see: the rough assignment has much higher energy.

👀 Takeaway: spectral embeddings keep connected neighbors close by minimizing graph roughness.

## 🔴 Advanced

### Advanced 1 — Sweep sigma in the affinity kernel

**Goal.** Compare kernel widths, because sigma controls how quickly distance fades into weak affinity. We build it in 3 steps.

In [ ]:
X_a1 = np.array([[0.0], [1.0], [2.0], [5.0]])
sigmas_a1 = np.array([0.5, 1.0, 2.0, 4.0])
weights_a1 = []
for sigma_a1 in sigmas_a1:
    A_a1 = knn_affinity(X_a1, k=2, sigma=sigma_a1)
    weights_a1.append(A_a1[0, 1])
print("edge 0-1 weights:", np.round(weights_a1, 3))

▶ What you'll see: the same distance receives larger weight as sigma grows.

In [ ]:
lambda2_a1 = []
for sigma_a1 in sigmas_a1:
    vals_a1 = np.linalg.eigvalsh(laplacian(knn_affinity(X_a1, k=2, sigma=sigma_a1)))
    lambda2_a1.append(vals_a1[1])
print("second eigenvalues:", np.round(lambda2_a1, 3))
assert len(lambda2_a1) == 4

▶ What you'll see: the graph's low-frequency spectrum changes as affinities soften.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sigmas_a1, lambda2_a1, marker="o", color="purple")
plt.title("Advanced 1: sigma changes graph connectivity strength")
plt.xlabel("sigma")
plt.ylabel("second-smallest eigenvalue")
plt.show()

▶ What you'll see: larger sigma generally strengthens weak long edges and changes spectral gaps.

👀 Takeaway: sigma is a locality knob, so it should be inspected rather than accepted blindly.

### Advanced 2 — Compare raw and standardized embeddings

**Goal.** Standardize features before graph building, because unscaled features can dominate distances and distort the map. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[0.0, 0.0], [0.0, 2.0], [1.0, 0.2], [1.0, 2.2]])
X_big_a2 = X_a2.copy()
X_big_a2[:, 0] *= 10.0
A_raw_a2 = knn_affinity(X_big_a2, k=1, sigma=3.0)
print("raw scaled A edges:", int(np.sum(A_raw_a2 > 0) / 2))

▶ What you'll see: the graph is built after feature 0 has been artificially magnified.

In [ ]:
mean_a2 = X_big_a2.mean(axis=0)
std_a2 = X_big_a2.std(axis=0)
X_std_a2 = (X_big_a2 - mean_a2) / std_a2
A_std_a2 = knn_affinity(X_std_a2, k=1, sigma=3.0)
print("standardized feature means:", np.round(X_std_a2.mean(axis=0), 6))
assert np.allclose(X_std_a2.mean(axis=0), 0.0)

▶ What you'll see: standardization centers both features and puts them on comparable units.

In [ ]:
vals_raw_a2, Z_raw_a2 = spectral_coordinates(A_raw_a2, dim=2)
vals_std_a2, Z_std_a2 = spectral_coordinates(A_std_a2, dim=2)
print("raw eigenvalues:", np.round(vals_raw_a2, 3))
print("std eigenvalues:", np.round(vals_std_a2, 3))

▶ What you'll see: changing scale can change graph weights and therefore the spectral coordinates.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(Z_raw_a2[:, 0], Z_raw_a2[:, 1], s=90, color="red")
ax[0].set_title("raw scaled")
ax[1].scatter(Z_std_a2[:, 0], Z_std_a2[:, 1], s=90, color="teal")
ax[1].set_title("standardized")
plt.suptitle("Advanced 2: preprocessing changes the map")
plt.show()

▶ What you'll see: the two embeddings can organize points differently because their graphs differ.

👀 Takeaway: preprocessing is part of the unsupervised model, not a neutral prelude.

### Advanced 3 — Align two embeddings before comparing stability

**Goal.** Compare embeddings up to sign and rotation, because eigenvector coordinates are not unique in orientation. We build it in 4 steps.

In [ ]:
X_a3 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
A_a3 = knn_affinity(X_a3, k=2, sigma=2.0)
_, Z_a3 = spectral_coordinates(A_a3, dim=2)
Z_flip_a3 = Z_a3 @ np.array([[0.0, 1.0], [1.0, 0.0]])
print("original shape:", Z_a3.shape, "flipped shape:", Z_flip_a3.shape)
assert Z_a3.shape == Z_flip_a3.shape

▶ What you'll see: both embeddings have the same information but swapped coordinate axes.

In [ ]:
U_a3, _, Vt_a3 = np.linalg.svd(Z_flip_a3.T @ Z_a3)
R_a3 = U_a3 @ Vt_a3
Z_aligned_a3 = Z_flip_a3 @ R_a3
err_a3 = float(np.linalg.norm(Z_aligned_a3 - Z_a3))
print("alignment error:", round(err_a3, 6))
assert err_a3 < 1e-10

▶ What you'll see: after orthogonal alignment, the swapped embedding matches the original.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_a3[:, 0], Z_a3[:, 1], s=90, label="original")
plt.scatter(Z_aligned_a3[:, 0], Z_aligned_a3[:, 1], marker="x", s=90, label="aligned")
plt.title("Advanced 3: compare after alignment")
plt.legend()
plt.show()

▶ What you'll see: the aligned crosses sit on top of the original points.

👀 Takeaway: stability checks should compare geometry, not arbitrary eigenvector orientation.

### Advanced 4 — Bootstrap a graph stability score

**Goal.** Resample points and compare neighbor edges, because fragile graphs produce fragile embeddings. We build it in 4 steps.

In [ ]:
X_a4 = np.array([[0.0, 0.0], [0.1, 0.0], [1.0, 0.0], [1.1, 0.0], [3.0, 0.0], [3.1, 0.0]])
A_ref_a4 = knn_affinity(X_a4, k=1, sigma=1.0) > 0
rng_a4 = np.random.default_rng(0)
print("reference edges:", int(A_ref_a4.sum() / 2))

▶ What you'll see: the reference graph links close pairs.

In [ ]:
scores_a4 = []
for t_a4 in range(20):
    noise_a4 = 0.03 * rng_a4.normal(size=X_a4.shape)
    A_noisy_a4 = knn_affinity(X_a4 + noise_a4, k=1, sigma=1.0) > 0
    both_a4 = np.logical_and(A_ref_a4, A_noisy_a4).sum() / 2
    either_a4 = np.logical_or(A_ref_a4, A_noisy_a4).sum() / 2
    scores_a4.append(both_a4 / either_a4)
print("stability scores:", np.round(scores_a4[:5], 3), "...")
assert min(scores_a4) >= 0.0 and max(scores_a4) <= 1.0

▶ What you'll see: each score is a Jaccard overlap between the original and noisy edge sets.

In [ ]:
mean_score_a4 = float(np.mean(scores_a4))
print("mean stability:", round(mean_score_a4, 3))
assert mean_score_a4 > 0.8

▶ What you'll see: this well-separated toy graph is highly stable to small noise.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(scores_a4, marker="o", color="seagreen")
plt.ylim(0, 1.05)
plt.title("Advanced 4: neighbor-graph stability")
plt.xlabel("noise trial")
plt.ylabel("edge Jaccard score")
plt.show()

▶ What you'll see: scores stay near 1 when local neighbor relationships are robust.

👀 Takeaway: stability checks turn the warning about fragile unsupervised outputs into a measurable diagnostic.

### Advanced 5 — Use a fallback when the graph is disconnected

**Goal.** Detect disconnected graphs before interpreting coordinates, because disconnected components make relative positions arbitrary. We build it in 4 steps.

In [ ]:
X_a5 = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 0.0], [5.0, 1.0]])
A_sparse_a5 = knn_affinity(X_a5, k=1, sigma=2.0)
vals_sparse_a5 = np.linalg.eigvalsh(laplacian(A_sparse_a5))
components_sparse_a5 = int(np.sum(np.isclose(vals_sparse_a5, 0.0, atol=1e-8)))
print("sparse components:", components_sparse_a5)
assert components_sparse_a5 == 2

▶ What you'll see: the nearest-neighbor graph splits into two components.

In [ ]:
A_connected_a5 = knn_affinity(X_a5, k=2, sigma=4.0)
vals_connected_a5 = np.linalg.eigvalsh(laplacian(A_connected_a5))
components_connected_a5 = int(np.sum(np.isclose(vals_connected_a5, 0.0, atol=1e-8)))
print("connected components:", components_connected_a5)
assert components_connected_a5 == 1

▶ What you'll see: increasing k and sigma creates enough bridge weight to connect the graph.

In [ ]:
_, Z_sparse_a5 = spectral_coordinates(A_sparse_a5, dim=2)
_, Z_connected_a5 = spectral_coordinates(A_connected_a5, dim=2)
print("sparse Z shape:", Z_sparse_a5.shape, "connected Z shape:", Z_connected_a5.shape)
assert Z_sparse_a5.shape == Z_connected_a5.shape == (4, 2)

▶ What you'll see: both produce coordinates, but only the connected graph has a meaningful global relationship.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(Z_sparse_a5[:, 0], Z_sparse_a5[:, 1], s=90, color="red")
ax[0].set_title("disconnected")
ax[1].scatter(Z_connected_a5[:, 0], Z_connected_a5[:, 1], s=90, color="teal")
ax[1].set_title("connected")
plt.suptitle("Advanced 5: connectivity check before interpretation")
plt.show()

▶ What you'll see: the connected graph gives one shared coordinate system, while disconnected pieces have arbitrary relative placement.

👀 Takeaway: always diagnose components before reading global meaning into an unsupervised map.